### Visualizing Embeddings from Presentation Slides
In this Notebook, three different types of Embeddings (text, visual, mixed) are compared and tested, in terms of their suitability to cluster Presentation slides based on their content. 
The data is stored via [Huggingface](https://huggingface.co/datasets/ScaDS-AI/SlightInsight_Cache).

In [ ]:
import sys
import os

# Add the root directory to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [ ]:
from caching import load_full_hf_cache
import pandas as pd
import dask.array as da
import dask
import requests
import numpy as np
import stackview
from io import BytesIO
from pdf2image import convert_from_bytes
import dask.array as da
import numpy as np
import umap.umap_ as umap
import os
from PIL import Image

repo_name = "ScaDS-AI/SlideInsight_Cache_v2"
df = load_full_hf_cache(repo_name=repo_name)

In [ ]:
df.head()

#### Perform UMAP for each Embeddingtype for each Slide

In [ ]:
# Initialize UMAP reducer
reducer = umap.UMAP(n_components=2, random_state=42)

# Initialize a dictionary to store UMAP results
umap_results = {}

# Apply UMAP to each embedding type
for embedding_type in ['text_embedding', 'visual_embedding', 'mixed_embedding']:
    # Convert embeddings to a numpy array
    embeddings = np.array(df[embedding_type].tolist())
    
    # Apply UMAP
    umap_embeddings = reducer.fit_transform(embeddings)
    
    # Store results in the dictionary
    umap_results[embedding_type] = {
        'UMAP0': umap_embeddings[:, 0],
        'UMAP1': umap_embeddings[:, 1]
    }

# Create a new DataFrame to hold UMAP results for each embedding type
df_umap = df.copy()

for embedding_type, umap_data in umap_results.items():
    df_umap[f"{embedding_type}_UMAP0"] = umap_data['UMAP0']
    df_umap[f"{embedding_type}_UMAP1"] = umap_data['UMAP1']

# Output the DataFrame with UMAP results
df_umap.head()

In [ ]:
import matplotlib.pyplot as plt

# Define embedding types for visualization
embedding_types = ['text_embedding', 'visual_embedding', 'mixed_embedding']

# Plot UMAP results for each embedding type
plt.figure(figsize=(12, 4))

for i, embedding_type in enumerate(embedding_types, 1):
    plt.subplot(1, 3, i)
    plt.scatter(df_umap[f"{embedding_type}_UMAP0"], df_umap[f"{embedding_type}_UMAP1"], alpha=0.5)
    plt.xlabel("UMAP0")
    plt.ylabel("UMAP1")
    plt.title(f"UMAP of {embedding_type.capitalize()} Embeddings")

plt.tight_layout()
plt.show()


## Use this Image Dataset to match image of Slides and Embeddings to visualize the whole dataset using sliceplot

In [ ]:
from datasets import load_dataset
import numpy as np
from tqdm import tqdm

dataset_name = "ScaDS-AI/Slide_Insight_Images_v2"
dataset = load_dataset(dataset_name, split="train", streaming=True)

keys = df["key"].values
matched_data = [] 

for sample in tqdm(dataset, desc="Processing and resizing images"):
    key = sample["key"]

    if key in keys:
        img = sample["image"]
        w, h = img.size
        scale = 384 / min(w, h)
        img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)

        img_array = np.array(img)

        matched_data.append({"key": key, "image": img_array})

# Convert to DataFrame
df_images = pd.DataFrame(matched_data)

In [ ]:
# Merge embeddings DataFrame (`df_umap`) with images DataFrame (`df_images`) based on "key"
df_merged = df_umap.merge(df_images, on="key", how="inner")  # Keep only matching keys

### Process Image Data

In [ ]:
def pad_image(img_array, target_height, target_width):
    height, width, channels = img_array.shape
    padded = np.zeros((target_height, target_width, channels), dtype=img_array.dtype)
    # Center the image
    pad_top = (target_height - height) // 2
    pad_left = (target_width - width) // 2
    padded[pad_top:pad_top+height, pad_left:pad_left+width, :] = img_array
    return padded

max_height = max(img.shape[0] for img in df_merged["image"])
max_width = max(img.shape[1] for img in df_merged["image"])

# Pad all images
padded_images = [
    pad_image(img, max_height, max_width) for img in df_merged["image"]
]

# Now safely stack to NumPy array
images_np = np.array(padded_images)

In [ ]:
# Visualize with stackview
stackview.sliceplot(
    df_merged,
    images_np,
    column_x="mixed_embedding_UMAP0",
    column_y="mixed_embedding_UMAP1",
    zoom_factor=1,
    zoom_spline_order=2
)

# Visualize using Wordcloud Plot

In [ ]:
stackview.wordcloudplot(df_umap, column_text="extracted_text", column_x="mixed_embedding_UMAP0", column_y="mixed_embedding_UMAP1")

### remove the "largest" words, that don't refer to any content topic, but rather to authors, fill words, etc.

In [ ]:
import re

def clean_text(text, large_words):
    text = text.lower()  # Convert text to lowercase for uniformity
    
    # Remove multi-word phrases first
    for phrase in sorted(large_words, key=len, reverse=True):  # Sort by length to prevent partial replacements
        text = re.sub(r'\b' + re.escape(phrase.lower()) + r'\b', '', text)

    # Remove single letters (except 'I' and 'a' which are valid words)
    text = re.sub(r'\b[a-zA-Z]\b', '', text) 

    # Remove any extra spaces created by removals
    text = re.sub(r'\s+', ' ', text).strip()

    return text
    
large_words = ['Robert Haase', 'haesleinhuepf', 'Haase', 'haesleinhuepf BIDS', 'robert haase', 'welcome', 'town', 'et al', 'i3d bio', 'logo small', 'use', 'cid', 'dataweekleipzig april',
               'haase haesleinhuepf', 'bids lecture', 'haesleinhuepf bids', 'der', 'die', 'und', 'josh moore', 'meeting josh','moore welcome', 'bioimage town', 'https', 'meeting welcome', 'hands meeting']
large_words_lo = [word.lower() for word in large_words]

df_umap["cleaned_text"] = df_umap["extracted_text"].apply(lambda x: clean_text(x, large_words_lo))

df_umap.head()

### Plot Wordcloud with the cleaned Text

In [ ]:
stackview.wordcloudplot(df_umap, column_text="cleaned_text", column_x="text_embedding_UMAP0", column_y="text_embedding_UMAP1")

### create new embeddings for the cleaned text

In [ ]:
from sentence_transformers import SentenceTransformer

text_model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1")

def calculate_cleaned_text_embedding(df):
    df["cleaned_text_embedding"] = df["cleaned_text"].apply(lambda text: text_model.encode(text))
    return df

df_cleaned_umap = calculate_cleaned_text_embedding(df_umap)
df_cleaned_umap.head()

### calculate a new UMAP

In [ ]:
# Initialize a dictionary to store UMAP results
cleaned_umap_results = {}
embeddings = np.array(df_cleaned_umap['cleaned_text_embedding'].tolist())

# Apply UMAP
cleaned_umap_embeddings = reducer.fit_transform(embeddings)

# Store results in the dictionary
cleaned_umap_results['cleaned_text_embedding'] = {
    'UMAP0': cleaned_umap_embeddings[:, 0],
    'UMAP1': cleaned_umap_embeddings[:, 1]
}

for embedding_type, umap_data in cleaned_umap_results.items():
    df_cleaned_umap["cleaned_text_embedding_UMAP0"] = umap_data['UMAP0']
    df_cleaned_umap["cleaned_text_embedding_UMAP1"] = umap_data['UMAP1']

# Output the DataFrame with UMAP results
df_cleaned_umap.head()

## Plot the Wordcloud again, using the cleaned text and corresponding embeddings

In [ ]:
stackview.wordcloudplot(df_cleaned_umap, column_text="cleaned_text", column_x="cleaned_text_embedding_UMAP0", column_y="cleaned_text_embedding_UMAP1")

## Plot the whole Stackview Plot again

In [ ]:
# Merge dataframes
df_merged = df_umap.merge(df_images, on="key", how="inner")  # Keep only matching keys

In [ ]:
stackview.sliceplot(
    df_merged,
    images_np,
    column_x="cleaned_text_embedding_UMAP0",
    column_y="cleaned_text_embedding_UMAP1",
    zoom_factor=1,
    zoom_spline_order=2
)

In [ ]:
from sentence_transformers import SentenceTransformer

text_model = SentenceTransformer("Alibaba-NLP/gte-multilingual-base", trust_remote_code=True)

def calculate_cleaned_text_embedding(df):
    df["cleaned_text_embedding_gte"] = df["cleaned_text"].apply(lambda text: text_model.encode(text))
    return df

calculate_cleaned_text_embedding(df_umap)
df_umap.head()